# Automated Article Search for a Systematic Review (OpenAlex)


1. Searches OpenAlex for ten query phrases.
2. Records the total number of results for each query and the number of records retrieved.
3. Retrieves up to the top 200 records for each query (two pages of 100).
4. Extracts the title, authors, year, DOI, abstract, journal/conference, and links.
5. Merges the records and removes duplicates by DOI, then OpenAlex ID, and then title.
6. Creates the CSV files, the Excel file, the human screening template, and the search report, and delivers them as a ZIP package.

**Methodological note:** This is a new supplementary search in OpenAlex; do not confuse it with the historical searches in Google Scholar, Semantic Scholar, arXiv, and ACL Anthology. The total number of API results differs from the number of retrieved records and the number of unique records. Final screening must be done by a human, and the final PRISMA requires the actual counts from all sources.

There is no need to install Python on your computer; run it in Colab.

## 1) Install Libraries and Settings

In [1]:
!pip -q install pandas requests openpyxl

import re, time, zipfile, requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
# OUT = Path("/content/openalex_review_search")
OUT = Path("OpenAlex_Results")
# OUT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

START_DATE = "2023-01-01"
END_DATE = "2026-11-13"
CAP_PER_QUERY = 200
PER_PAGE = 100
API = "https://api.openalex.org/works"
RUN_UTC = datetime.now(timezone.utc).isoformat(timespec="seconds")

QUERIES = [
    ("Q1", 'LLM knowledge distillation graph'),
    ("Q2", 'large language model graph neural network distillation'),
    ("Q3", 'language model graph knowledge transfer'),
    ("Q4", 'pretrained language model graph knowledge distillation'),
    ("Q5", 'language model teacher graph student'),
    ("Q6", 'LLM-guided graph knowledge transfer'),
    ("Q7", 'LLM soft-label distillation graph'),
    ("Q8", 'LLM rationale distillation graph learning'),
    ("Q9", 'temporal graph knowledge distillation language model'),
    ("Q10", '(LLM OR "large language model" OR PLM OR "pretrained language model") AND (distillation OR distilled OR "knowledge transfer") AND (graph OR GNN OR "graph neural network")'),
]
print("Ready to run; number of queries:", len(QUERIES))


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Ready to run; number of queries: 10


## 2) Functions for Retrieval, Abstract Extraction, and Normalization

In [2]:
def api_get(params, retries=5):
    headers = {"User-Agent": "AcademicLiteratureReviewSearch/1.0"}
    for attempt in range(retries):
        try:
            r = requests.get(API, params=params, headers=headers, timeout=60)
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(min(2**attempt, 30))
                continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if attempt == retries - 1:
                raise
            time.sleep(min(2**attempt, 30))
    raise RuntimeError("OpenAlex did not respond after several attempts.")

def abstract_text(index):
    if not isinstance(index, dict):
        return ""
    pos = {}
    for word, indexes in index.items():
        for i in indexes:
            pos[int(i)] = word
    return " ".join(pos[i] for i in sorted(pos))

def norm_doi(x):
    x = (x or "").strip().lower()
    return re.sub(r"^https?://(dx\.)?doi\.org/", "", x)

def norm_title(x):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", (x or "").lower())).strip()

def unpack(work, qid, query, page):
    authors = []
    for a in work.get("authorships", []) or []:
        name = (a.get("author") or {}).get("display_name")
        if name: authors.append(name)
    loc = work.get("primary_location") or {}
    source = loc.get("source") or {}
    oa = work.get("open_access") or {}
    best = work.get("best_oa_location") or {}
    return {
        "openalex_id": work.get("id", ""),
        "doi": norm_doi(work.get("doi", "")),
        "title": work.get("title", "") or "",
        "publication_year": work.get("publication_year"),
        "publication_date": work.get("publication_date", ""),
        "authors": "; ".join(authors),
        "abstract": abstract_text(work.get("abstract_inverted_index")),
        "document_type": work.get("type", ""),
        "venue": source.get("display_name", "") or "",
        "landing_page_url": loc.get("landing_page_url", "") or "",
        "best_oa_url": best.get("landing_page_url", "") or "",
        "is_oa": oa.get("is_oa"),
        "oa_status": oa.get("oa_status", ""),
        "cited_by_count": work.get("cited_by_count"),
        "query_id": qid, "query_text": query, "retrieved_page": page
    }

## 3) Running the Searches
For each query, `meta.count` is recorded first, and then at most two pages are retrieved.

In [3]:
records, logs = [], []
for qid, query in QUERIES:
    print(f"Searching {qid}: {query}")
    params = {
        "search": query,
        "filter": f"from_publication_date:{START_DATE},to_publication_date:{END_DATE}",
        "per-page": PER_PAGE, "page": 1
    }
    first = api_get(params)
    total = (first.get("meta") or {}).get("count", 0)
    fetched = 0
    pages = 1 if total <= PER_PAGE else 2
    for page in range(1, pages + 1):
        data = first if page == 1 else api_get({**params, "page": page})
        for w in data.get("results", []) or []:
            if fetched >= CAP_PER_QUERY: break
            records.append(unpack(w, qid, query, page))
            fetched += 1
        time.sleep(0.15)
    logs.append({
        "query_id": qid, "query_text": query, "source": "OpenAlex Works API",
        "run_timestamp_utc": RUN_UTC, "date_from": START_DATE, "date_to": END_DATE,
        "total_hits_reported_by_api": total, "retrieval_cap": CAP_PER_QUERY,
        "records_retrieved": fetched, "pages_retrieved": pages, "status": "completed"
    })
    print(f"  Total API results: {total:,} | Retrieved: {fetched:,}")

raw = pd.DataFrame(records)
log = pd.DataFrame(logs)
print("Total raw records:", len(raw))
display(log)

Searching Q1: LLM knowledge distillation graph
  Total API results: 20,111 | Retrieved: 200
Searching Q2: large language model graph neural network distillation
  Total API results: 35,553 | Retrieved: 200
Searching Q3: language model graph knowledge transfer
  Total API results: 188,164 | Retrieved: 200
Searching Q4: pretrained language model graph knowledge distillation
  Total API results: 21,458 | Retrieved: 200
Searching Q5: language model teacher graph student
  Total API results: 69,114 | Retrieved: 200
Searching Q6: LLM-guided graph knowledge transfer
  Total API results: 45,179 | Retrieved: 200
Searching Q7: LLM soft-label distillation graph
  Total API results: 5,251 | Retrieved: 200
Searching Q8: LLM rationale distillation graph learning
  Total API results: 4,266 | Retrieved: 200
Searching Q9: temporal graph knowledge distillation language model
  Total API results: 18,121 | Retrieved: 200
Searching Q10: (LLM OR "large language model" OR PLM OR "pretrained language model") 

,query_id,query_text,source,run_timestamp_utc,date_from,date_to,total_hits_reported_by_api,retrieval_cap,records_retrieved,pages_retrieved,status
0,Q1,LLM knowledge distillation graph,OpenAlex Works API,2026-11-14T16:23:23+00:00,2023-01-01,2026-11-13,20111,200,200,2,completed
1,Q2,large language model graph neural network distillation,OpenAlex Works API,2026-11-14T16:23:23+00:00,2023-01-01,2026-11-13,35553,200,200,2,completed
2,Q3,language model graph knowledge transfer,OpenAlex Works API,2026-11-14T16:23:23+00:00,2023-01-01,2026-11-13,188164,200,200,2,completed
3,Q4,pretrained language model graph knowledge distillation,OpenAlex Works API,2026-11-14T16:23:23+00:00,2023-01-01,2026-11-13,21458,200,200,2,completed
4,Q5,language model teacher graph student,OpenAlex Works API,2026-11-14T16:23:23+00:00,2023-01-01,2026-11-13,69114,200,200,2,completed
5,Q6,LLM-guided graph knowledge transfer,OpenAlex Works API,2026-11-14T16:23:23+00:00,2023-01-01,2026-11-13,45179,200,200,2,completed
6,Q7,LLM soft-label distillation graph,OpenAlex Works API,2026-11-14T16:23:23+00:00,2023-01-01,2026-11-13,5251,200,200,2,completed
7,Q8,LLM rationale distillation graph learning,OpenAlex Works API,2026-11-14T16:23:23+00:00,2023-01-01,2026-11-13,4266,200,200,2,completed
8,Q9,temporal graph knowledge distillation language model,OpenAlex Works API,2026-11-14T16:23:23+00:00,2023-01-01,2026-11-13,18121,200,200,2,completed
9,Q10,"(LLM OR ""large language model"" OR PLM OR ""pretrained language model"") AND (distillation OR distilled OR ""knowledge t...",OpenAlex Works API,2026-11-14T16:23:23+00:00,2023-01-01,2026-11-13,46188,200,200,2,completed


## 4) Merging and Removing Duplicates
Duplicate detection priority: DOI, OpenAlex ID, normalized title.

In [4]:
def deduplicate(df):
    if df.empty:
        return df.copy(), pd.DataFrame()
    d = df.copy()
    d["title_norm"] = d["title"].map(norm_title)
    def key(row):
        if row["doi"]: return "doi:" + row["doi"]
        if row["openalex_id"]: return "openalex:" + row["openalex_id"]
        return "title:" + row["title_norm"]
    d["dedup_key"] = d.apply(key, axis=1)
    keep, dup = [], []
    for k, g in d.groupby("dedup_key", sort=False):
        g = g.copy()
        g["_score"] = (g["abstract"].fillna("").str.len()
                       + 2*g["doi"].fillna("").astype(str).str.len()
                       + g["authors"].fillna("").str.len()
                       + g["venue"].fillna("").str.len())
        g = g.sort_values("_score", ascending=False)
        row = g.iloc[0].drop(labels=["_score"])
        row["matched_queries"] = "; ".join(sorted(set(g["query_id"].dropna())))
        row["number_of_query_hits"] = len(g)
        keep.append(row)
        for _, x in g.iloc[1:].iterrows():
            dup.append({
                "dedup_key": k, "kept_title": row["title"],
                "duplicate_title": x["title"], "doi": x["doi"],
                "openalex_id": x["openalex_id"],
                "duplicate_query_id": x["query_id"],
                "duplicate_query_text": x["query_text"]
            })
    unique = pd.DataFrame(keep).drop(columns=["title_norm", "dedup_key"], errors="ignore")
    return unique, pd.DataFrame(dup)

unique, duplicates = deduplicate(raw)
print(f"Raw: {len(raw):,} | Unique: {len(unique):,} | Duplicates removed: {len(raw)-len(unique):,}")
display(unique[["title", "publication_year", "doi", "matched_queries"]].head(10))

Raw: 2,000 | Unique: 1,434 | Duplicates removed: 566


,title,publication_year,doi,matched_queries
0,LLM-driven Knowledge Distillation for Dynamic Text-Attributed Graphs,2025,10.48550/arxiv.2502.10914,Q1; Q10; Q2; Q4; Q6; Q9
1,LLM-Guided Knowledge Distillation for Temporal Knowledge Graph Reasoning,2026,10.48550/arxiv.2602.14428,Q1; Q5; Q6; Q9
2,PRISM: Reliable LLM Knowledge Distillation from Graph-Context Perspectives for Text-Attributed Graph Learning,2026,10.1145/3799682.3840050,Q1; Q10; Q8
3,DRAG: Distilling RAG for SLMs from LLMs to Transfer Knowledge and Mitigate Hallucination via Evidence and Graph-base...,2025,10.48550/arxiv.2506.01954,Q1; Q6; Q8
4,MedCoach: Enhancing Medical Reasoning in LLMs via Knowledge Graph-Augmented Chain-of-Thought Distillation,2026,10.18653/v1/2026.findings-acl.1683,Q1; Q7; Q8
5,DRAG: Distilling RAG for SLMs from LLMs to Transfer Knowledge and Mitigate Hallucination via Evidence and Graph-base...,2025,10.18653/v1/2025.acl-long.358,Q1; Q6; Q8
6,SynthLineage: Submodular-Guided Corpus Distillation and Knowledge Graph Provenance for Collapse-Resistant LLM Fine-T...,2026,10.1109/iri69576.2026.00092,Q1
7,Large Language Model Meets Graph Neural Network in Knowledge Distillation,2025,10.1609/aaai.v39i16.33901,Q1; Q10; Q2; Q3; Q5; Q8
8,LLM and GNN Are Complementary: Distilling LLM for Multimodal Graph Learning,2025,10.1109/bigdata66926.2025.11402034,Q1; Q10; Q2
9,Contextualization Distillation from Large Language Model for Knowledge Graph Completion,2024,10.18653/v1/2024.findings-eacl.32,Q1; Q2; Q3; Q4; Q5; Q7; Q8; Q9


## 5) ASReview-Ready File and Human Screening Template

In [5]:
cols = ["title", "abstract", "authors", "publication_year", "doi",
        "openalex_id", "venue", "landing_page_url", "matched_queries"]
asreview = unique.reindex(columns=cols).copy()
screening = asreview.copy()
for col in ["title_abstract_decision", "title_abstract_exclusion_reason",
            "full_text_retrieved", "full_text_decision",
            "full_text_exclusion_reason", "reviewer", "screening_notes"]:
    screening[col] = ""
display(asreview.head())

,title,abstract,authors,publication_year,doi,openalex_id,venue,landing_page_url,matched_queries
0,LLM-driven Knowledge Distillation for Dynamic Text-Attributed Graphs,"Dynamic Text-Attributed Graphs (DyTAGs) have numerous real-world applications, e.g. social, collaboration, citation,...","Roy, Amit; Ning Yan; Masood Mortazavi",2025,10.48550/arxiv.2502.10914,https://openalex.org/W4407687339,arXiv (Cornell University),http://arxiv.org/abs/2502.10914,Q1; Q10; Q2; Q4; Q6; Q9
1,LLM-Guided Knowledge Distillation for Temporal Knowledge Graph Reasoning,"Temporal knowledge graphs (TKGs) support reasoning over time-evolving facts, yet state-of-the-art models are often c...",Wang Xing; Wei Song; Siyu Lin; Chen Wu; Man Wang,2026,10.48550/arxiv.2602.14428,https://openalex.org/W7129717845,arXiv (Cornell University),http://arxiv.org/abs/2602.14428,Q1; Q5; Q6; Q9
2,PRISM: Reliable LLM Knowledge Distillation from Graph-Context Perspectives for Text-Attributed Graph Learning,Text-attributed graph (TAG) learning aims to leverage both textual attributes and graph structures for node classifi...,Jung-Hyun Kim; So-Bin Jung; David Yoon Suk Kang; Sang‐Wook Kim,2026,10.1145/3799682.3840050,https://openalex.org/W7225090758,ACM International Conference on Information and Knowledge Management (CIKM),https://doi.org/10.1145/3799682.3840050,Q1; Q10; Q8
3,DRAG: Distilling RAG for SLMs from LLMs to Transfer Knowledge and Mitigate Hallucination via Evidence and Graph-base...,Retrieval-Augmented Generation (RAG) methods have proven highly effective for tasks requiring factual consistency an...,"J. Chen; Aidar Myrzakhan; Luo, Yaxin; Khan, Hassaan Muhammad; Sondos Mahmoud Bsharat; Zhiqiang Shen",2025,10.48550/arxiv.2506.01954,https://openalex.org/W4414899016,arXiv (Cornell University),http://arxiv.org/abs/2506.01954,Q1; Q6; Q8
4,MedCoach: Enhancing Medical Reasoning in LLMs via Knowledge Graph-Augmented Chain-of-Thought Distillation,"Despite the advanced capabilities of Large Language Models (LLMs), training specialized reasoning models for the med...",Chuan Li; Ye Lyu; Chengyu Wang; Mingyuan Fan; Cen Chen,2026,10.18653/v1/2026.findings-acl.1683,https://openalex.org/W7166813320,Findings of the Association for Computational Linguistics: ACL (Findings of ACL),https://doi.org/10.18653/v1/2026.findings-acl.1683,Q1; Q7; Q8


## 6) Preliminary PRISMA Counts (OpenAlex Only)
This cell only fills in the counts computed from the retrieved data. The screening-stage cells must be completed after human screening is done.

In [6]:
n_raw, n_unique = len(raw), len(unique)
prisma = pd.DataFrame([
    ["Records retrieved from OpenAlex before duplicate removal", n_raw],
    ["Duplicates removed in the OpenAlex set", n_raw-n_unique],
    ["Unique OpenAlex records for screening", n_unique],
    ["Records screened at title/abstract", ""],
    ["Records excluded at title/abstract", ""],
    ["Full texts assessed", ""],
    ["Full texts excluded", ""],
    ["Studies included in the final review", ""],
], columns=["Stage", "Count"])
display(prisma)
print("This table covers only the supplementary OpenAlex search, not other historical sources.")

,Stage,Count
0,Records retrieved from OpenAlex before duplicate removal,2000
1,Duplicates removed in the OpenAlex set,566
2,Unique OpenAlex records for screening,1434
3,Records screened at title/abstract,
4,Records excluded at title/abstract,
5,Full texts assessed,
6,Full texts excluded,
7,Studies included in the final review,


This table covers only the supplementary OpenAlex search, not other historical sources.


## 7) Saving CSV, Excel, and ZIP

In [7]:
# Search methodology report
report = [
    "OPENALEX SUPPLEMENTARY SEARCH REPORT",
    f"Run timestamp UTC: {RUN_UTC}",
    f"Publication date range: {START_DATE} to {END_DATE}",
    f"Per-query retrieval cap: {CAP_PER_QUERY}",
    f"Raw retrieved records: {len(raw)}",
    f"Unique records: {len(unique)}",
    f"Duplicates removed: {len(raw)-len(unique)}",
    "",
    "This is a supplementary OpenAlex search, not a reconstruction of historical searches in Google Scholar, Semantic Scholar, arXiv, or ACL Anthology.",
    "API total_hits, records retrieved, and unique records are distinct counts.",
    "", "Queries:"
] + [f"{qid}: {q}" for qid, q in QUERIES]

files_out = {
    "OpenAlex_Search_Log.csv": log,
    "OpenAlex_Retrieved_Records_Raw.csv": raw,
    "OpenAlex_Unique_Records.csv": unique,
    "OpenAlex_Duplicate_Report.csv": duplicates,
    "ASReview_Import.csv": asreview,
    "Human_Screening_Template.csv": screening,
    "PRISMA_Counts_OpenAlex_Only.csv": prisma,
}
for name, frame in files_out.items():
    frame.to_csv(OUT/name, index=False, encoding="utf-8-sig")
(OUT/"OpenAlex_Search_Report.txt").write_text("\\n".join(report), encoding="utf-8")

xlsx = OUT/"OpenAlex_Search_and_Screening.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
    for sheet, frame in [
        ("Search_Log", log), ("Raw_Records", raw), ("Unique_Records", unique),
        ("Duplicates", duplicates), ("ASReview_Import", asreview),
        ("Human_Screening", screening), ("PRISMA_Counts", prisma)
    ]:
        frame.to_excel(writer, sheet_name=sheet, index=False)

# zip_path = Path("/content/OpenAlex_Review_Search_Results.zip")
zip_path = Path("OpenAlex_Review_Search_Results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file(): z.write(p, arcname=p.name)

print("Output files:")
for p in sorted(OUT.iterdir()):
    print(p.name, f"({p.stat().st_size:,} bytes)")
print("ZIP package:", zip_path)

Output files:
ASReview_Import.csv (2,327,302 bytes)
Human_Screening_Template.csv (2,337,487 bytes)
OpenAlex_Duplicate_Report.csv (186,239 bytes)
OpenAlex_Retrieved_Records_Raw.csv (3,512,489 bytes)
OpenAlex_Search_and_Screening.xlsx (4,038,799 bytes)
OpenAlex_Search_Log.csv (1,685 bytes)
OpenAlex_Search_Report.txt (1,083 bytes)
OpenAlex_Unique_Records.csv (2,520,423 bytes)
PRISMA_Counts_OpenAlex_Only.csv (324 bytes)
ZIP package: OpenAlex_Review_Search_Results.zip
